# De Novo Peptide Sequencing — offline plots

Sanity-check notebook over `denovo.db`. Mirrors the interactive Quarto site (`index.qmd`); use this for ad-hoc SQL exploration or to confirm what the published site is rendering. PNGs in `plots/` are **not committed** — the Quarto site is canonical.

In [ ]:
import sqlite3
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import PathPatch, Wedge
from matplotlib.path import Path as MplPath
from matplotlib.lines import Line2D
import numpy as np
import networkx as nx
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
conn = sqlite3.connect("denovo.db")

## 1. Database overview

In [ ]:
tables = ["algorithm", "algorithm_repository", "publication", "author",
          "affiliation", "country", "city", "author_affiliation",
          "publication_author", "publication_algorithm", "publication_citation",
          "journal_impact"]
for t in tables:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn).iloc[0, 0]
    print(f"{t:25s} {n:>6d} rows")

In [ ]:
# Catalog of algorithms — repository URLs come from the normalized
# algorithm_repository table (1:N), concatenated with whitespace.
df_alg = pd.read_sql("""
    SELECT a.name, a.algorithm_family, a.kind, a.is_deep_learning AS is_dl,
           a.acquisition_mode AS acq, a.aliases,
           (SELECT GROUP_CONCAT(ar.url, ' ')
              FROM (SELECT url FROM algorithm_repository
                    WHERE algorithm_id = a.id ORDER BY sort_order) ar) AS repository
    FROM algorithm a
    ORDER BY a.name
""", conn)
df_alg

## 2. Hero counters

Same scalars the site renders in the top stat strip + the elevator-pitch sentence.

In [ ]:
df_pub = pd.read_sql(
    """SELECT id, title, publication_date AS date, publication_type AS type,
              publisher, journal, doi, url, version
       FROM publication
       ORDER BY publication_date""",
    conn, parse_dates=["date"]
)
df_pub["year"] = df_pub["date"].dt.year

# DL-flag join through publication_algorithm so we can find the earliest DL paper.
df_alg_lookup = pd.read_sql("SELECT id, name, is_deep_learning FROM algorithm", conn)
df_pa = pd.read_sql("SELECT publication_id, algorithm_id FROM publication_algorithm", conn)
df_pub_dl = (df_pa.merge(df_alg_lookup, left_on="algorithm_id", right_on="id")
                  .groupby("publication_id")["is_deep_learning"].max()
                  .reset_index().rename(columns={"publication_id": "id"}))
df_pub_w = df_pub.merge(df_pub_dl, on="id", how="left")
first_dl_year = int(df_pub_w[df_pub_w["is_deep_learning"] == 1]["year"].min())

n_papers   = len(df_pub)
n_models   = pd.read_sql("SELECT COUNT(*) n FROM algorithm", conn).iloc[0, 0]
n_authors  = pd.read_sql("SELECT COUNT(*) n FROM author", conn).iloc[0, 0]
n_countries= pd.read_sql("""SELECT COUNT(DISTINCT c.id) n
                            FROM country c JOIN affiliation a ON a.country_id=c.id""", conn).iloc[0, 0]
first_year, last_year = int(df_pub["year"].min()), int(df_pub["year"].max())
n_preprints     = (df_pub["type"] == "preprint").sum()
n_peer_reviewed = (df_pub["type"] == "peer-reviewed").sum()

print(f"{'papers':>14s}: {n_papers}")
print(f"{'methods':>14s}: {n_models}")
print(f"{'authors':>14s}: {n_authors}")
print(f"{'countries':>14s}: {n_countries}")
print(f"{'years covered':>14s}: {first_year}-{last_year}  (first DL paper: {first_dl_year})")
print(f"{'preprints':>14s}: {n_preprints}")
print(f"{'peer-reviewed':>14s}: {n_peer_reviewed}")

## 3. The wave — publications per year, stacked by type

In [ ]:
df_wave = (df_pub.dropna(subset=["year"])
                 .assign(year=lambda d: d["year"].astype(int))
                 .groupby(["year", "type"]).size().unstack(fill_value=0))
type_order = [t for t in ["preprint", "peer-reviewed", "ML conference", "thesis", "commentary"]
              if t in df_wave.columns]
df_wave = df_wave[type_order]
fig, ax = plt.subplots(figsize=(12, 5))
df_wave.plot(kind="bar", stacked=True, ax=ax,
             color=sns.color_palette("Blues_r", len(type_order)),
             edgecolor="white")
ax.set_xlabel("Year"); ax.set_ylabel("Papers")
ax.set_title(f"De novo peptide sequencing papers per year — {n_papers} total")
ax.legend(title="Type", loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

## 4. Classification breakdowns

Three classifier columns: `kind`, `is_deep_learning`, `acquisition_mode`.

In [ ]:
df_alg_class = pd.read_sql(
    """SELECT kind, is_deep_learning AS is_dl, acquisition_mode AS acq, COUNT(*) AS n
       FROM algorithm GROUP BY kind, is_deep_learning, acquisition_mode""", conn)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (col, title) in zip(axes, [("kind", "Kind"), ("is_dl", "Deep learning?"), ("acq", "Acquisition")]):
    s = df_alg_class.groupby(col, dropna=False)["n"].sum().sort_values(ascending=True)
    s.index = s.index.fillna("—")
    ax.barh(s.index.astype(str), s.values, color=sns.color_palette("Set2", len(s)))
    ax.set_title(title); ax.set_xlabel("Algorithms")
plt.tight_layout(); plt.show()

## 5. Architectures timeline

Horizontal swim lanes per `algorithm_family`. Each method shows as a dot at its first publication date. Mirrors the Quarto chart but rendered statically.

In [ ]:
band_defs = [
    ("Heuristic",         "#8a96a0", 1.0),
    ("Graph / DP",        "#6c757d", 2.5),
    ("HMM",               "#4d6a8c", 1.0),
    ("Decision tree",     "#7b6f43", 1.0),
    ("Random Forest",     "#a5673f", 1.2),
    ("Learning-to-rank",  "#5f6b7a", 1.2),
    ("CNN + RNN",         "#4C72B0", 2.0),
    ("Transformer (AR)",  "#DD8452", 8.0),
    ("GNN",               "#937860", 1.2),
    ("CNN",               "#8172B3", 1.2),
    ("Transformer (NAR)", "#55A868", 2.5),
    ("Diffusion",         "#C44E52", 1.2),
    ("Flow",              "#937DC2", 1.2),
]
BAND_GAP = 0.15
bands, y_cursor = {}, 0.0
for name, color, h in band_defs:
    bot, top = y_cursor, y_cursor + h
    bands[name] = (bot, top, (bot + top) / 2, color, h)
    y_cursor = top + BAND_GAP

df_innov = pd.read_sql("""
    SELECT a.name, a.algorithm_family AS family, a.short_description AS desc,
           MIN(p.publication_date) AS first_pub
    FROM algorithm a
    JOIN publication_algorithm pa ON a.id = pa.algorithm_id
    JOIN publication p ON pa.publication_id = p.id
    WHERE a.algorithm_family IS NOT NULL AND p.publication_date IS NOT NULL
    GROUP BY a.id ORDER BY first_pub
""", conn, parse_dates=["first_pub"])

Y_TIERS = [0.5, 0.8, 0.2, 0.9, 0.1, 0.7, 0.3, 0.6, 0.4,
           0.85, 0.15, 0.95, 0.05, 0.75, 0.25, 0.65, 0.35, 0.55, 0.45,
           0.88, 0.12, 0.78, 0.22, 0.58, 0.42]
MIN_GAP_DAYS = 360
items, last_at_tier = [], {}
for _, row in df_innov.iterrows():
    fam = row["family"]
    if fam not in bands: continue
    date = row["first_pub"]
    placed = False
    for ti, frac in enumerate(Y_TIERS):
        k = (fam, ti)
        if k not in last_at_tier or (date - last_at_tier[k]).days >= MIN_GAP_DAYS:
            last_at_tier[k] = date
            items.append((date, row["name"], fam, frac)); placed = True; break
    if not placed:
        items.append((date, row["name"], fam, 0.5))

fig, ax = plt.subplots(figsize=(28, 14))
for bname, (bot, top, _, color, _) in bands.items():
    ax.axhspan(bot, top, color=color, alpha=0.09, zorder=0)
    ax.axhline(bot, color=color, lw=0.4, alpha=0.3)
ax.set_yticks([b[2] for b in bands.values()])
ax.set_yticklabels(list(bands.keys()), fontsize=12, fontweight="bold")
for lbl, color in zip(ax.get_yticklabels(), [b[3] for b in bands.values()]):
    lbl.set_color(color)

for date, name, fam, frac in items:
    bot, top, center, color, bh = bands[fam]
    y = bot + frac * bh
    ax.plot(date, y, "o", color=color, ms=10, mec="white", mew=1.3, zorder=4)
    va, dy = ("bottom", 8) if frac >= 0.5 else ("top", -8)
    ax.annotate(name, xy=(date, y), xytext=(0, dy), textcoords="offset points",
                fontsize=9, fontweight="bold", va=va, ha="center", color=color)

ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.tick_params(axis="y", length=0)
ax.set_xlabel("First publication →", fontsize=13)
ax.set_title("Architectures timeline", fontsize=18, fontweight="bold", pad=18)
ax.yaxis.grid(False)
sns.despine(left=True)
plt.tight_layout(); plt.show()

## 6. Where the work happens — equirectangular world scatter

Static analogue of the Quarto site's d3 world map. Each city plotted at its lat/lng with a circle proportional to the number of distinct authors affiliated there. No country borders are drawn (no cartopy / geopandas dependency) — it's a sanity-check, not a publication chart.

In [ ]:
df_cities = pd.read_sql("""
    SELECT ci.id, ci.name AS city, co.name AS country, ci.lat, ci.lng,
           COUNT(DISTINCT aa.author_id)     AS authors,
           COUNT(DISTINCT af.id)            AS affiliations
    FROM city ci
    JOIN country co ON co.id = ci.country_id
    LEFT JOIN affiliation af       ON af.city_id = ci.id
    LEFT JOIN author_affiliation aa ON aa.affiliation_id = af.id
    WHERE ci.lat IS NOT NULL
    GROUP BY ci.id ORDER BY authors DESC
""", conn)

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_facecolor("#f6f8fa")
# Faint grid lines mimicking parallels/meridians.
for lat in range(-60, 90, 30): ax.axhline(lat, color="#dadfe5", lw=0.5, zorder=1)
for lon in range(-180, 181, 30): ax.axvline(lon, color="#dadfe5", lw=0.5, zorder=1)

size_scale = 600 / np.sqrt(df_cities["authors"].max())
sc = ax.scatter(df_cities["lng"], df_cities["lat"],
                s=np.sqrt(df_cities["authors"]) * size_scale,
                c=df_cities["authors"], cmap="Blues",
                edgecolor="#1f4ec7", linewidth=0.6, alpha=0.78, zorder=3)
for _, row in df_cities.head(10).iterrows():
    ax.text(row["lng"]+1.5, row["lat"]+1.5, row["city"], fontsize=8, color="#1d2330")
ax.set_xlim(-180, 180); ax.set_ylim(-60, 80)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title(f"Cities — {len(df_cities)} geocoded research locations · circle size ∝ distinct authors")
plt.colorbar(sc, ax=ax, shrink=0.7, label="distinct authors")
plt.tight_layout(); plt.show()

## 7. Top countries & top institutions

In [ ]:
df_geo = pd.read_sql("""
    SELECT c.name AS country,
           COUNT(DISTINCT af.id)        AS affiliations,
           COUNT(DISTINCT aa.author_id) AS authors
    FROM country c
    JOIN affiliation af ON af.country_id = c.id
    LEFT JOIN author_affiliation aa ON aa.affiliation_id = af.id
    GROUP BY c.id ORDER BY authors DESC
""", conn)
df_inst = pd.read_sql("""
    SELECT af.name AS institution, c.name AS country,
           COUNT(DISTINCT aa.author_id) AS authors
    FROM affiliation af
    JOIN country c ON af.country_id = c.id
    JOIN author_affiliation aa ON aa.affiliation_id = af.id
    GROUP BY af.name ORDER BY authors DESC LIMIT 15
""", conn)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].barh(df_geo["country"][::-1], df_geo["authors"][::-1],
             color=sns.color_palette("coolwarm", len(df_geo)))
axes[0].set_xlabel("Distinct authors"); axes[0].set_title("Authors per country")

labels = [f"{r['institution']} ({r['country']})" for _, r in df_inst.iterrows()]
axes[1].barh(labels[::-1], df_inst["authors"][::-1],
             color=sns.color_palette("rocket", len(df_inst)))
axes[1].set_xlabel("Distinct authors"); axes[1].set_title("Top 15 institutions")
plt.tight_layout(); plt.show()

## 8. Top authors

In [ ]:
df_top_authors = pd.read_sql("""
    SELECT a.name, COUNT(pa.publication_id) AS publications
    FROM author a JOIN publication_author pa ON a.id = pa.author_id
    GROUP BY a.id ORDER BY publications DESC LIMIT 20
""", conn)
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_top_authors["name"][::-1], df_top_authors["publications"][::-1],
        color=sns.color_palette("viridis", len(df_top_authors)))
ax.set_xlabel("Number of publications"); ax.set_title("Top 20 authors")
plt.tight_layout(); plt.show()

## 9. Co-authorship network — authors with ≥ 3 papers

Same prolific-author filter as the Quarto site. Pie-chart nodes for multi-affiliation authors; edges tinted by shared affiliation.

In [ ]:
df_coauth = pd.read_sql("""
    SELECT pa1.author_id AS a1, pa2.author_id AS a2, COUNT(*) AS weight
    FROM publication_author pa1
    JOIN publication_author pa2 ON pa1.publication_id = pa2.publication_id
    WHERE pa1.author_id < pa2.author_id
    GROUP BY pa1.author_id, pa2.author_id
""", conn)
author_names = dict(pd.read_sql("SELECT id, name FROM author", conn).values)
G = nx.Graph()
for _, row in df_coauth.iterrows():
    G.add_edge(author_names[row["a1"]], author_names[row["a2"]], weight=row["weight"])
prolific = {r[0] for r in pd.read_sql("""
    SELECT a.name FROM author a JOIN publication_author pa ON a.id = pa.author_id
    GROUP BY a.id HAVING COUNT(*) >= 3
""", conn).values}
G_sub = G.subgraph([n for n in G.nodes if n in prolific]).copy()

df_author_aff = pd.read_sql("""
    SELECT a.name AS author_name, aff.name AS affiliation
    FROM author a JOIN author_affiliation aa ON a.id = aa.author_id
    JOIN affiliation aff ON aa.affiliation_id = aff.id
    GROUP BY a.id, aff.id
""", conn)
author_all_affs = {}
for name in G_sub.nodes:
    affs = list(df_author_aff[df_author_aff["author_name"] == name]["affiliation"].unique())
    author_all_affs[name] = affs if affs else ["Unknown"]

author_primary_aff = {}
for name in G_sub.nodes:
    affs = author_all_affs[name]
    if len(affs) == 1:
        author_primary_aff[name] = affs[0]; continue
    neighbors = list(G_sub.neighbors(name))
    best_aff, best_count = affs[0], -1
    for aff in affs:
        c = sum(1 for nb in neighbors if aff in author_all_affs.get(nb, []))
        if c > best_count:
            best_aff, best_count = aff, c
    author_primary_aff[name] = best_aff

unique_affs = sorted({a for v in author_all_affs.values() for a in v})
palette = sns.color_palette("tab20", n_colors=len(unique_affs))
aff_color_map = dict(zip(unique_affs, palette))

# Force-directed layout, one component at a time, packed horizontally.
pos = {}; cursor_x, GAP = 0.0, 0.15
for comp_nodes in sorted(nx.connected_components(G_sub), key=len, reverse=True):
    comp = G_sub.subgraph(comp_nodes).copy()
    G_layout = comp.copy()
    for n in comp.nodes:
        a = author_primary_aff[n]
        for other in comp.nodes:
            if other != n and author_primary_aff[other] == a:
                if G_layout.has_edge(n, other):
                    G_layout[n][other]["weight"] += 8
                else:
                    G_layout.add_edge(n, other, weight=8)
    p = nx.spring_layout(G_layout, k=0.8, seed=42, iterations=200, weight="weight")
    xs, ys = [v[0] for v in p.values()], [v[1] for v in p.values()]
    xspan = max(max(xs)-min(xs), 1e-6); yspan = max(max(ys)-min(ys), 1e-6)
    scale = np.sqrt(len(comp_nodes)) / 6.0
    for n in p:
        pos[n] = np.array([(p[n][0]-min(xs))/xspan*scale + cursor_x,
                          (p[n][1]-min(ys))/yspan*scale])
    cursor_x += scale + GAP
ymid = (max(v[1] for v in pos.values()) + min(v[1] for v in pos.values())) / 2
for n in pos: pos[n][1] -= ymid

fig, ax = plt.subplots(figsize=(20, 10))
deg = dict(G_sub.degree())
max_deg = max(deg.values()) or 1
x_all = [v[0] for v in pos.values()]
R = (max(x_all) - min(x_all)) * 0.016
edge_colors = [aff_color_map[author_primary_aff[u]] if author_primary_aff[u]==author_primary_aff[v] else "#cccccc"
               for u, v in G_sub.edges]
edge_weights = [G_sub[u][v]["weight"] for u, v in G_sub.edges]
nx.draw_networkx_edges(G_sub, pos, alpha=0.25, ax=ax,
                       width=[w*0.6 for w in edge_weights], edge_color=edge_colors)
for name in G_sub.nodes:
    x, y = pos[name]; affs = author_all_affs[name]
    r = R * (0.6 + 0.4 * deg[name] / max_deg)
    if len(affs) == 1:
        ax.add_patch(plt.Circle((x, y), r, fc=aff_color_map[affs[0]], ec="white", lw=1.2, alpha=0.92, zorder=3))
    else:
        for i, a in enumerate(affs):
            ax.add_patch(Wedge((x, y), r, i*360/len(affs), (i+1)*360/len(affs),
                               fc=aff_color_map[a], ec="white", lw=0.8, alpha=0.92, zorder=3))
nx.draw_networkx_labels(G_sub, pos, font_size=7, ax=ax)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"Co-authorship network — {G_sub.number_of_nodes()} authors (≥ 3 papers), "
             f"{G_sub.number_of_edges()} edges", fontsize=13)
plt.tight_layout(); plt.show()

## 10. Authors ↔ models bipartite network

Same prolific-author filter as the co-authorship graph; an edge connects an author to every algorithm they helped publish on. Diamonds are algorithms (colored by family); circles are authors.

In [ ]:
df_aa_alg = pd.read_sql("""
    SELECT a.name AS author, alg.name AS algo, alg.algorithm_family AS family,
           COUNT(*) AS weight
    FROM publication_author pa
    JOIN author a ON a.id = pa.author_id
    JOIN publication_algorithm pal ON pal.publication_id = pa.publication_id
    JOIN algorithm alg ON alg.id = pal.algorithm_id
    WHERE a.id IN (SELECT author_id FROM publication_author
                   GROUP BY author_id HAVING COUNT(*) >= 3)
    GROUP BY a.id, alg.id
""", conn)

family_color = {
    "CNN + RNN": "#4C72B0", "Transformer (AR)": "#DD8452", "GNN": "#937860",
    "CNN": "#8172B3", "Transformer (NAR)": "#55A868", "Diffusion": "#C44E52",
    "Flow": "#937DC2", "Graph / DP": "#6c757d", "HMM": "#4d6a8c",
    "Heuristic": "#8a96a0", "Decision tree": "#7b6f43",
    "Random Forest": "#a5673f", "Learning-to-rank": "#5f6b7a",
}
B = nx.Graph()
for _, row in df_aa_alg.iterrows():
    B.add_node(row["author"], kind="author")
    B.add_node(row["algo"],   kind="algo", family=row["family"])
    B.add_edge(row["author"], row["algo"], weight=row["weight"])

pos = nx.spring_layout(B, k=0.8, seed=7, iterations=200)
fig, ax = plt.subplots(figsize=(18, 10))
deg = dict(B.degree())
max_deg = max(deg.values()) or 1
for u, v, d in B.edges(data=True):
    ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]],
            color="#bbb", lw=0.5 + 0.4*d["weight"], alpha=0.4, zorder=1)
for n, attrs in B.nodes(data=True):
    x, y = pos[n]
    r = 0.015 + 0.04 * np.sqrt(deg[n] / max_deg)
    if attrs["kind"] == "author":
        ax.add_patch(plt.Circle((x, y), r, fc="#cdd6e0", ec="#8a94a3", lw=0.7, zorder=3))
    else:
        # Diamond polygon scaled by radius
        diamond = plt.Polygon([(x, y+r), (x+r, y), (x, y-r), (x-r, y)],
                              fc=family_color.get(attrs.get("family") or "", "#999"),
                              ec="#222", lw=0.7, zorder=3)
        ax.add_patch(diamond)
        ax.text(x, y - r - 0.015, n, fontsize=7, ha="center", va="top", fontweight="bold", color="#222")
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Authors ↔ models — diamonds are algorithms colored by family, circles are prolific authors",
             fontsize=13)
plt.tight_layout(); plt.show()

## 11. Venues — papers per journal + 2yr OpenAlex citedness

In [ ]:
df_venues = pd.read_sql("""
    SELECT journal AS venue, publication_type AS type, COUNT(*) AS papers
    FROM publication
    WHERE journal IS NOT NULL AND journal != ''
    GROUP BY journal, publication_type
    ORDER BY papers DESC
""", conn)
top_venues = (df_venues.groupby("venue")["papers"].sum().sort_values(ascending=False).head(15))
df_if = pd.read_sql("SELECT journal, two_yr_citedness FROM journal_impact", conn)
df_if = df_if.set_index("journal")["two_yr_citedness"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh(top_venues.index[::-1], top_venues.values[::-1],
             color=sns.color_palette("mako", len(top_venues)))
axes[0].set_xlabel("Papers"); axes[0].set_title("Top 15 venues (by paper count)")

df_if_top = df_if.dropna().sort_values(ascending=False).head(15)
axes[1].barh(df_if_top.index[::-1], df_if_top.values[::-1],
             color=sns.color_palette("flare", len(df_if_top)))
axes[1].set_xlabel("OpenAlex 2-year mean citedness"); axes[1].set_title("Top 15 venues (by OpenAlex IF₂yr)")
plt.tight_layout(); plt.show()

## 12. Preprint → peer-reviewed lifecycle gap

For every (method, version) pair where both a preprint and a peer-reviewed publication exist, the gap (in months) between them. Mirrors the lifecycle bar chart on the Quarto site.

In [ ]:
# Map each publication to a single representative algorithm name.
df_alg_first = pd.read_sql("""
    SELECT pa.publication_id, MIN(alg.name) AS method
    FROM publication_algorithm pa JOIN algorithm alg ON alg.id = pa.algorithm_id
    GROUP BY pa.publication_id
""", conn)
df_life = df_pub.merge(df_alg_first, left_on="id", right_on="publication_id", how="left")
df_life["version"] = df_life["version"].fillna("")
df_life["group"]   = df_life["method"].fillna("") + "|" + df_life["version"]

pairs = []
for grp, sub in df_life.groupby("group"):
    method = sub["method"].iloc[0]
    if not method: continue
    version = sub["version"].iloc[0]
    pre = sub[sub["type"] == "preprint"]["date"].min()
    peer = sub[sub["type"].isin(["peer-reviewed", "ML conference"])]["date"].min()
    if pd.notna(pre) and pd.notna(peer) and peer >= pre:
        gap = (peer - pre).days / 30.44
        pairs.append({"method": method, "version": version, "pre": pre, "peer": peer, "gap_months": gap})
df_life_pairs = pd.DataFrame(pairs).sort_values("gap_months", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(6, len(df_life_pairs) * 0.28)))
label = df_life_pairs.apply(lambda r: f"{r['method']} {r['version']}".strip(), axis=1)
ax.barh(label, df_life_pairs["gap_months"], color=sns.color_palette("viridis", len(df_life_pairs)))
ax.set_xlabel("Months between preprint and peer-reviewed publication")
ax.set_title(f"Preprint → peer-reviewed lifecycle — {len(df_life_pairs)} paired methods")
median = df_life_pairs["gap_months"].median()
ax.axvline(median, color="#cf222e", lw=1.2, ls="--")
ax.text(median, len(df_life_pairs)*0.98, f"  median {median:.1f} mo", color="#cf222e", va="top")
plt.tight_layout(); plt.show()

## 13. Citation arc diagram

Papers placed left→right by publication date and stratified vertically by `kind`. Each arc connects a citing paper to a paper it cites, curving upward. Mirrors the d3 chart on the Quarto site.

In [ ]:
df_cit = pd.read_sql("SELECT citing_id, cited_id, source FROM publication_citation", conn)
df_pub_kind = pd.read_sql("""
    SELECT p.id, p.publication_date AS date, p.title,
           (SELECT alg.kind FROM publication_algorithm pal
             JOIN algorithm alg ON alg.id = pal.algorithm_id
            WHERE pal.publication_id = p.id LIMIT 1) AS kind,
           (SELECT GROUP_CONCAT(alg.name, ', ') FROM publication_algorithm pal
             JOIN algorithm alg ON alg.id = pal.algorithm_id
            WHERE pal.publication_id = p.id) AS models
    FROM publication p
""", conn, parse_dates=["date"])

KIND_ORDER = ["meta", "benchmark", "review", "adjacent",
              "downstream-application", "post-processor", "algorithm"]
KIND_COLOR = {
    "algorithm":              "#1f6feb",
    "post-processor":         "#bf8700",
    "downstream-application": "#1a7f37",
    "adjacent":               "#a371f7",
    "review":                 "#cf222e",
    "benchmark":              "#0969da",
    "meta":                   "#8c959f",
}
involved = set(df_cit["citing_id"]) | set(df_cit["cited_id"])
df_nodes = df_pub_kind[df_pub_kind["id"].isin(involved)].copy()
df_nodes["kind"] = df_nodes["kind"].fillna("meta")
df_nodes["row"] = df_nodes["kind"].map(lambda k: KIND_ORDER.index(k) if k in KIND_ORDER else len(KIND_ORDER)-1)

in_deg = df_cit.groupby("cited_id").size()
df_nodes["in_deg"] = df_nodes["id"].map(in_deg).fillna(0).astype(int)
max_in = max(df_nodes["in_deg"].max(), 1)
# Within each row, the most-cited papers sit highest; others spread below.
row_h = 1.0
df_nodes["y"] = df_nodes["row"] * row_h + (1 - np.sqrt(df_nodes["in_deg"] / max_in)) * row_h * 0.8 + 0.1

pos = {row["id"]: (mdates.date2num(row["date"]), row["y"]) for _, row in df_nodes.iterrows()}

fig, ax = plt.subplots(figsize=(18, 10))
for ki, kind in enumerate(KIND_ORDER):
    ax.axhspan(ki*row_h, (ki+1)*row_h, color=KIND_COLOR[kind], alpha=0.05, zorder=0)
    ax.text(ax.get_xlim()[0] if ax.get_xlim()[0] != 0 else mdates.date2num(df_nodes["date"].min()),
            ki*row_h + row_h/2, kind, ha="right", va="center",
            color=KIND_COLOR[kind], fontweight="bold", fontsize=10)

for _, e in df_cit.iterrows():
    if e["citing_id"] not in pos or e["cited_id"] not in pos: continue
    (x1, y1), (x2, y2) = pos[e["citing_id"]], pos[e["cited_id"]]
    lift = max(0.2, min(2.0, abs(x1 - x2) * 0.004))
    cy = min(y1, y2) - lift
    path = MplPath([(x1, y1), ((x1+x2)/2, cy), ((x1+x2)/2, cy), (x2, y2)],
                   [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4])
    ax.add_patch(PathPatch(path, fc="none", ec="#94a3b8", lw=0.4, alpha=0.18, zorder=1))

ax.scatter([p[0] for p in pos.values()], [p[1] for p in pos.values()],
           s=[(3 + 5*np.sqrt(d/max_in))**2 for d in df_nodes["in_deg"]],
           c=[KIND_COLOR.get(k, "#999") for k in df_nodes["kind"]],
           edgecolor="white", linewidth=0.5, zorder=3)

ax.invert_yaxis()  # algorithms at the bottom, headroom for arcs at top
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_yticks([])
ax.set_title(f"Citation flow — {len(df_nodes)} papers, {len(df_cit)} intra-catalog citations (citing → cited)")
sns.despine(left=True)
plt.tight_layout(); plt.show()

## 14. Custom queries

In [ ]:
# Example: all publications a specific author appears on.
pd.read_sql("""
    SELECT p.title, p.publication_date,
           (SELECT GROUP_CONCAT(alg.name, ', ') FROM publication_algorithm pal
             JOIN algorithm alg ON alg.id=pal.algorithm_id
            WHERE pal.publication_id=p.id) AS methods
    FROM author a
    JOIN publication_author pa ON a.id = pa.author_id
    JOIN publication p ON pa.publication_id = p.id
    WHERE a.name = 'Siqi Sun'
    ORDER BY p.publication_date DESC
""", conn)

In [ ]:
# Example: every affiliation for a given author.
pd.read_sql("""
    SELECT a.name AS author, af.name AS institution, af.department, c.name AS country
    FROM author a
    JOIN author_affiliation aa ON a.id = aa.author_id
    JOIN affiliation af ON aa.affiliation_id = af.id
    JOIN country c ON af.country_id = c.id
    WHERE a.name = 'Xiang Zhang'
""", conn)

In [ ]:
conn.close()